# CUPED and CUPAC in A/B tests

[All Quick Tutorials](README.md)

Compare an unadjusted A/B test with CUPED and CUPAC on the same simulated sample. Assign the roles each method needs, pass the settings to `ABTest`, and read the variance reduction and confidence intervals.

> **Backend requirement.** CUPED and CUPAC require the pandas backend.

## Contents

- [How a covariate reduces variance](#how-adjustment-works)
  - [Adjustment coefficient and variance](#cup-covariate-formula)
  - [Historical metric versus model prediction](#cup-covariate-choice)
  - [Overall mean and differences between groups](#cup-mean-preservation)
- [Prepare the data and roles](#data-and-roles)
  - [Simulate randomized assignment](#cup-simulate-assignment)
  - [Assign roles to outcomes and historical features](#cup-historical-roles)
- [Estimate the unadjusted A/B effect](#baseline)
- [Apply CUPED with a historical metric](#cuped)
  - [Configure `cuped_features`](#cuped-map-history)
  - [Read CUPED variance reduction](#cuped-report)
- [Apply CUPAC with model predictions](#cupac)
  - [Train on historical lags and predict the current outcome](#cupac-train-predict)
  - [Enable CUPAC with a Ridge model](#cupac-single-model)
  - [Compare historical and realized variance reduction](#cupac-variance-report)
  - [Inspect feature importance](#cupac-feature-importance)
  - [Try several CUPAC models](#cupac-models)
- [Compare the results](#compare)
  - [Compare effect estimates and interval widths](#cup-compare-intervals)
  - [Read the Holm-adjusted p-values](#cup-holm)

<a id="how-adjustment-works"></a>

## How a covariate reduces variance

<a id="cup-covariate-formula"></a>

### Adjustment coefficient and variance

Let $Y$ be the outcome and $C$ a covariate measured or constructed from pretreatment data. If $\operatorname{Var}(C)>0$, the coefficient $\theta$ below minimizes the variance of the adjusted metric.

$$
Y_{\mathrm{adj}}=Y-\theta(C-\mathbb E[C]),\qquad
\theta=\frac{\operatorname{Cov}(Y,C)}{\operatorname{Var}(C)}.
$$

At the optimal $\theta$ for a fixed covariate, $\operatorname{Var}(Y_{\mathrm{adj}})=\operatorname{Var}(Y)(1-\rho_{Y,C}^2)$ in the population. Stronger correlation with the outcome allows greater variance reduction. With randomized assignment, you can compare the adjusted group means if treatment does not affect the covariate. [Deng et al. on CUPED](https://ai.stanford.edu/~ronnyk/2013-02CUPEDImprovingSensitivityOfControlledExperiments.pdf)

<a id="cup-covariate-choice"></a>

### Historical metric versus model prediction

**CUPED** uses a previous value of the metric, $C=Y_{-1}$. **CUPAC** uses a prediction, $C=\widehat m(X_{-1},Y_{-1})$, from a model trained on historical data. Use only pretreatment inputs for prediction, excluding the current outcome, the assignment, and any features affected by treatment. [DoorDash on CUPAC](https://careersatdoordash.com/blog/improving-experimental-power-through-control-using-predictions-as-covariate-cupac/)

<a id="cup-mean-preservation"></a>

### Overall mean and differences between groups

On the same sample, HypEx centers the covariate at the overall mean $\bar C$. Thus $\bar Y_{\mathrm{adj}}=\bar Y$ up to floating-point rounding. For the difference between treatment and control, $\Delta\bar Y_{\mathrm{adj}}=\Delta\bar Y-\widehat\theta\,\Delta\bar C$. Random assignment makes the expected difference in the historical covariate zero, although a given sample can have chance imbalance.

<a id="data-and-roles"></a>

## Prepare the data and roles

<a id="cup-simulate-assignment"></a>

### Simulate randomized assignment

Generate 1,000 observations with an effect size of 0.1 and a seed of 42.

For this A/B example, simulate full compliance with randomized assignment `z`. In the generator, actual exposure `d` also depends on the latent variable `U`, which shifts `X1`. The simulated data contain both potential outcomes, so set `y` to `y0` when `z=0` and to `y1` when `z=1`.

In [1]:
import pandas as pd

from hypex import ABTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils.tutorial_data_creation import DataGenerator

generator = DataGenerator(
    n_samples=1_000,
    distributions={
        "X1": {"type": "normal", "mean": 1, "std": 1},
        "X2": {"type": "bernoulli", "p": 0.5},
        "y0": {"type": "normal", "mean": 1, "std": 5},
    },
    time_correlations={"X1": 0.2, "X2": 0.1, "y0": 0.8},
    effect_size=0.1,
    seed=42,
)
frame = generator.generate()
frame = frame.rename(columns={"y0_lag_1": "y_lag1", "y0_lag_2": "y_lag2"})
frame["y"] = frame["y0"].where(frame["z"] == 0, frame["y1"])
frame[["z", "y", "y_lag1", "y_lag2", "X1_lag1", "X1_lag2"]].head(3)

,z,y,y_lag1,y_lag2,X1_lag1,X1_lag2
0,1,1.558344,1.764458,3.405340,-0.056265,-0.192286
1,0,5.642941,2.601936,0.003204,1.349086,1.640866
2,1,6.419831,1.204498,-1.688911,0.060244,0.155618


`y_lag1` records the metric in the most recent pretreatment period. `y_lag2` records it one period earlier. `X1` and `X2` also have columns for both periods.

<a id="cup-historical-roles"></a>

### Assign roles to outcomes and historical features

Assign `TreatmentRole` to `z` and `TargetRole` to the outcome. `cofounders` names the underlying CUPAC features. `parent` and `lag` connect each historical column to its variable and period. CUPED needs the mapping from `y` to `y_lag1`.

This CUPAC example uses both historical periods and `cofounders`. The unadjusted A/B test needs `z` and `y`. CUPED adds only the historical column `y_lag1`.

In [2]:
roles = {
    "z": TreatmentRole(),
    "y": TargetRole(cofounders=["X1", "X2"]),
    "y_lag1": PreTargetRole(parent="y", lag=1),
    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X2_lag1": FeatureRole(parent="X2", lag=1),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
}
data = Dataset(roles=roles, data=frame[list(roles)])
{name: data.roles[name] for name in ("y_lag1", "X1_lag2")}

{'y_lag1': PreTarget(data_type=<class 'float'>, parent='y', lag=1),
 'X1_lag2': Feature(data_type=<class 'float'>, parent='X1', lag=2)}

HypEx recognizes `y_lag1` as lag 1 of `y` and `X1_lag2` as lag 2 of `X1`. These links let CUPAC assemble training and prediction data in time order.

<a id="baseline"></a>

## Estimate the unadjusted A/B effect

Run `ABTest()` on the original outcome as a baseline for comparing effect estimates and interval widths.

In [3]:
baseline = ABTest().execute(data)
baseline.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,y,1,0.651179,1.103755,0.452575,69.500841,-0.15769,1.06284,NOT OK,0.146387


In the `y` row, treatment minus control is about 0.453. The interval from −0.158 to 1.063 includes zero, and `TTest p-value` is about 0.1464.

<a id="cuped"></a>

## Apply CUPED with a historical metric

<a id="cuped-map-history"></a>

### Configure `cuped_features`

Use `cuped_features` to map the outcome to its historical covariate. HypEx runs the same A/B test on the original and adjusted outcomes.

The mapping identifies the covariate. You can leave `parent`, `lag`, and the CUPAC model options unset for this run.

In [4]:
cuped = ABTest(cuped_features={"y": "y_lag1"}).execute(data)
cuped.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,y,1,0.651179,1.103755,0.452575,69.500841,-0.157690,1.062840,NOT OK,0.146387
1,y_cuped,1,0.671685,1.083001,0.411316,61.236507,0.044636,0.777997,OK,0.028136


The estimate for `y_cuped` is about 0.411, with a confidence interval from 0.045 to 0.778 and a `TTest p-value` of about 0.0281. If the CUPED-adjusted outcome was chosen in advance as a standalone test, the result is significant at 0.05. The `y` row shows the original comparison.

<a id="cuped-report"></a>

### Read CUPED variance reduction

Open the CUPED report to see how much the metric variance changed.

In [5]:
cuped.cuped.variance_reductions

,feature,variance_reduction_pct
0,y_cuped,63.808189


`variance_reduction_pct` shows that the metric variance fell by about 63.8%. The comparison below measures the change in confidence interval width separately.

<a id="cupac"></a>

## Apply CUPAC with model predictions

<a id="cupac-train-predict"></a>

### Train on historical lags and predict the current outcome

HypEx trains a model to predict $Y_{-1}$ from $(X_{-2},Y_{-2})$. It then applies the model to $(X_{-1},Y_{-1})$ and uses that prediction as the covariate for $Y$.

<a id="cupac-single-model"></a>

### Enable CUPAC with a Ridge model

Enable this step with `enable_cupac=True`. The optional `cupac_models` accepts a model name or a list of candidates. With `None`, HypEx considers the models available in the current installation. Use `ridge` as the single candidate in this example.

This run uses the historical roles and can run independently of the CUPED example.

In [6]:
cupac = ABTest(
    enable_cupac=True,
    cupac_models="ridge",
).execute(data)
cupac.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,y,1,0.651179,1.103755,0.452575,69.500841,-0.157690,1.062840,NOT OK,0.146387
1,y_cupac,1,0.666001,1.088754,0.422753,63.476396,0.054444,0.791063,OK,0.024683


The `y_cupac` estimate is about 0.423, with a confidence interval from 0.054 to 0.791 and a `TTest p-value` of about 0.0247. As a standalone test, the CUPAC result is also significant at 0.05.

<a id="cupac-variance-report"></a>

### Compare historical and realized variance reduction

Open the CUPAC report to compare variance reduction in historical cross-validation with the reduction in the experiment sample.

In [7]:
cupac.cupac.variance_reductions

,target,best_model,variance_reduction_cv,variance_reduction_real
0,y,ridge,64.048727,63.475523


`best_model` shows `ridge`. `variance_reduction_cv` reports variance reduction in historical cross-validation, and `variance_reduction_real` reports it for the experiment sample. The latter is about 63.5%. The cross-validation estimate can vary slightly with the folds.

<a id="cupac-feature-importance"></a>

### Inspect feature importance

Inspect the features used by the selected model.

In [8]:
cupac.cupac.feature_importances

,target,feature,importance,model
0,y,X1_lag2,-0.197202,ridge
1,y,X2_lag2,-0.028106,ridge
2,y,y_lag2,0.808714,ridge


The feature names refer to the training period, so the report shows `X1_lag2`, `X2_lag2`, and `y_lag2`. For linear models, `importance` shows the signed coefficient averaged across cross-validation folds. Account for feature scales before ranking coefficients by magnitude.

<a id="cupac-models"></a>

### Try several CUPAC models

To compare models on the same history, pass a list to `cupac_models`. The base installation includes `linear`, `ridge`, and `lasso`. HypEx selects a model by variance reduction in historical cross-validation. Models with optional dependencies become available after installing the corresponding extra, such as `hypex[cat]` for CatBoost.

Use a list to supply several candidates or leave `cupac_models=None` to choose from the current installation's registry.

In [9]:
selected_cupac = ABTest(
    enable_cupac=True,
    cupac_models=["linear", "ridge", "lasso"],
).execute(data)
selected_cupac.cupac.variance_reductions

,target,best_model,variance_reduction_cv,variance_reduction_real
0,y,linear,64.051465,63.475226


`best_model` identifies the selected candidate. `variance_reduction_cv` refers to historical data, and `variance_reduction_real` to the observed experiment outcome. When candidates perform similarly, the choice can depend on the folds.

<a id="compare"></a>

## Compare the results

<a id="cup-compare-intervals"></a>

### Compare effect estimates and interval widths

Collect the effect estimate, interval width, and t-test p-value for each method. For CUPED and CUPAC, use the adjusted row for the outcome of interest.

The full `summary` reports appear above. Here, use `to_records()` to read the relevant rows and calculate interval widths.

In [10]:
def comparison_row(report, feature):
    row = next(item for item in report.to_records() if item["feature"] == feature)
    return {
        "difference": float(row["difference"]),
        "TTest p-value": float(row["TTest p-value"]),
        "interval width": float(row["ci upper"]) - float(row["ci lower"]),
    }


comparison = pd.DataFrame.from_dict(
    {
        "A/B": comparison_row(baseline.summary, "y"),
        "CUPED": comparison_row(cuped.summary, "y_cuped"),
        "CUPAC": comparison_row(cupac.summary, "y_cupac"),
    },
    orient="index",
)
comparison.round(4)

,difference,TTest p-value,interval width
A/B,0.4526,0.1464,1.2205
CUPED,0.4113,0.0281,0.7334
CUPAC,0.4228,0.0247,0.7366


The interval narrows from 1.221 to 0.733 with CUPED and to 0.737 with CUPAC. The difference in means also changes slightly because of chance imbalance in the covariate. For a real experiment, decide in advance which adjustment and which hypotheses will drive the decision.

<a id="cup-holm"></a>

### Read the Holm-adjusted p-values

The p-values above come from individual t-tests in `summary`. If you treat the original and adjusted outcomes as two hypotheses in the same family, read `multitest`, where HypEx applies Holm by default. The next cell shows the CUPED example.

In [11]:
cuped.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,y,TTest,0.14638715308650507,0.14638715308650507,1.0,False,1
1,y_cuped,TTest,0.02813604612009556,0.05627209224019112,0.5,False,1


For `y_cuped`, `new p-value` gives the adjusted p-value of about 0.0563. In this two-hypothesis family, it exceeds 0.05.

See [ABTest](ABTest.ipynb) for the basic interface and the [CUPED guide](../how_to_guide/cuped.ipynb) for a shorter standalone example.

If the experiment outcome is still pending, evaluate the models on historical data with the [virtual-target guide](../how_to_guide/cupac-history.ipynb).

To prepare a metric ahead of A/A or A/B, see how to [run the CUPED and CUPAC executors independently](../how_to_guide/manual-cuped-cupac.ipynb).